In [1]:
# 🧩 SAFETYEYE - Real-Time Inference (Oct 27 Task)
# ==========================================================
# Goal: Evaluate YOLOv8 PPE Detection model for real-time performance
# ==========================================================

# 1️⃣ Install Dependencies
!pip install -q ultralytics opencv-python torch matplotlib pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.7 MB/s eta 0:00:00


In [2]:
# 2️⃣ Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
# 3️⃣ Project Paths
import os

BASE_DIR = "/content/drive/MyDrive/SafetyEye"
MODEL_PATH = f"{BASE_DIR}/models/best_model/best.pt"
REALTIME_DIR = f"{BASE_DIR}/realtime_inference"
DATA_DIR = f"{REALTIME_DIR}/data"
OUTPUT_DIR = f"{REALTIME_DIR}/outputs"
SNAPSHOT_DIR = f"{OUTPUT_DIR}/snapshots"
REPORT_DIR = f"{REALTIME_DIR}/report"

# Create necessary folders
os.makedirs(SNAPSHOT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("✅ Paths ready")
print("Model:", MODEL_PATH)

✅ Paths ready
Model: /content/drive/MyDrive/SafetyEye/models/best_model/best.pt


In [25]:
# 4️⃣ Load YOLOv8 Model
from ultralytics import YOLO

model = YOLO(MODEL_PATH)
print("✅ Model loaded successfully!")
print("Classes:", model.names)

✅ Model loaded successfully!
Classes: {0: 'Hardhat', 1: 'Mask', 2: 'NO-Hardhat', 3: 'NO-Mask', 4: 'NO-Safety Vest', 5: 'Person', 6: 'Safety Cone', 7: 'Safety Vest', 8: 'machinery', 9: 'vehicle'}


In [30]:
# ==========================================================
# 🎥 5️⃣ Real-Time Video Inference with Metadata Logging
# ==========================================================
import cv2
import time
import numpy as np
import pandas as pd
import os
import glob

def process_video_realtime(video_path, output_dir, conf=0.5, imgsz=640, frame_save_interval=1.0):
    """
    Processes a video using YOLOv8 model and logs metadata.
    Saves snapshot frames when there is:
      - a significant detection change, OR
      - at least 1 second has passed since the last snapshot.
    Also logs frame-wise detections into a CSV file.
    """

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Could not open {video_path}")
        return None

    fps_in = cap.get(cv2.CAP_PROP_FPS) or 25.0
    w, h = int(cap.get(3)), int(cap.get(4))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out_path = os.path.join(output_dir, os.path.basename(video_path).replace(".mp4", "_realtime.mp4"))
    writer = cv2.VideoWriter(out_path, fourcc, fps_in, (w, h))

    frame_times = []
    frame_id = 0
    last_save_time = 0
    last_detections = None

    # For metadata logging
    metadata_records = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        start = time.time()
        results = model(frame, conf=conf, imgsz=imgsz)
        end = time.time()

        frame_time = end - start
        frame_times.append(frame_time)
        fps = 1.0 / frame_time if frame_time > 0 else 0

        annotated = results[0].plot()

        # Extract detection info
        current_boxes = results[0].boxes.xyxy.cpu().numpy() if results[0].boxes is not None else np.array([])
        current_conf = results[0].boxes.conf.cpu().numpy() if results[0].boxes is not None else np.array([])
        current_cls = results[0].boxes.cls.cpu().numpy() if results[0].boxes is not None else np.array([])

        # Log metadata per detection
        for i in range(len(current_boxes)):
            x1, y1, x2, y2 = current_boxes[i]
            metadata_records.append({
                "Video": os.path.basename(video_path),
                "Frame_ID": frame_id,
                "Class": model.names[int(current_cls[i])] if int(current_cls[i]) in model.names else "unknown",
                "Confidence": round(float(current_conf[i]), 3),
                "BBox": f"[{int(x1)}, {int(y1)}, {int(x2)}, {int(y2)}]",
                "FPS": round(fps, 2)
            })

        # Detection change logic
        detection_change = False
        if last_detections is None:
            detection_change = True
        else:
            if len(current_boxes) != len(last_detections):
                detection_change = True
            else:
                diff = np.abs(current_boxes - last_detections)
                if np.mean(diff) > 15:  # pixel threshold
                    detection_change = True

        # Save frame if detection change or 1s passed
        current_time = time.time()
        if detection_change or (current_time - last_save_time) >= frame_save_interval:
            snap_name = f"{os.path.basename(video_path).split('.')[0]}_frame_{frame_id}.jpg"
            snap_path = os.path.join(SNAPSHOT_DIR, snap_name)
            cv2.imwrite(snap_path, annotated)
            last_save_time = current_time
            last_detections = current_boxes

        # FPS overlay
        cv2.putText(annotated, f"FPS: {fps:.2f}", (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        writer.write(annotated)
        frame_id += 1

    cap.release()
    writer.release()

    avg_time = np.mean(frame_times)
    avg_fps = 1.0 / avg_time if avg_time > 0 else 0
    print(f"✅ Processed {video_path}")
    print(f"Average FPS: {avg_fps:.2f} | Avg Inference Time: {avg_time*1000:.1f} ms")

    # Save per-video metadata log
    metadata_df = pd.DataFrame(metadata_records)
    meta_path = os.path.join(REPORT_DIR, f"{os.path.basename(video_path).split('.')[0]}_metadata.csv")
    metadata_df.to_csv(meta_path, index=False)
    print(f"📄 Detection metadata saved to: {meta_path}")

    # Append FPS log
    log_path = os.path.join(output_dir, "fps_log.txt")
    with open(log_path, "a") as f:
        f.write(f"\nVideo: {video_path}\n")
        f.write(f"Average FPS: {avg_fps:.2f}\n")
        f.write(f"Average Inference Time: {avg_time*1000:.1f} ms\n")
        f.write(f"Total Frames: {frame_id}\n")

    return out_path, avg_fps, avg_time


# ==========================================================
# 6️⃣ Run on All Videos in /realtime_inference/data/
# ==========================================================
video_folder = f"{BASE_DIR}/realtime_inference/data"
video_files = glob.glob(f"{video_folder}/*.mp4")

if not video_files:
    print(f"⚠️ No video files found in {video_folder}")
else:
    results_summary = []

    for video_path in video_files:
        print(f"\n🎬 Processing {os.path.basename(video_path)} ...")
        result = process_video_realtime(video_path, OUTPUT_DIR)
        if result is not None:
            output_video, avg_fps, avg_time = result
            results_summary.append({
                "Video Name": os.path.basename(video_path),
                "Average FPS": round(avg_fps, 2),
                "Average Inference Time (ms)": round(avg_time * 1000, 1)
            })

    # Save all video results to CSV report
    if results_summary:
        df_summary = pd.DataFrame(results_summary)
        summary_path = os.path.join(REPORT_DIR, "realtime_inference_report.csv")
        df_summary.to_csv(summary_path, index=False)
        print("\n✅ All videos processed. Summary saved to:", summary_path)
        print(df_summary)



🎬 Processing site_demo_1.mp4 ...

0: 384x640 2 Hardhats, 3 Persons, 1 Safety Cone, 2 Safety Vests, 2 machinerys, 9.4ms
Speed: 3.7ms preprocess, 9.4ms inference, 6.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Hardhats, 4 Persons, 1 Safety Cone, 1 Safety Vest, 2 machinerys, 8.8ms
Speed: 3.2ms preprocess, 8.8ms inference, 4.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Hardhats, 3 Persons, 2 Safety Cones, 1 Safety Vest, 2 machinerys, 7.6ms
Speed: 4.3ms preprocess, 7.6ms inference, 4.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Hardhats, 4 Persons, 1 Safety Cone, 1 Safety Vest, 1 machinery, 9.5ms
Speed: 4.9ms preprocess, 9.5ms inference, 4.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Hardhats, 4 Persons, 2 Safety Cones, 1 Safety Vest, 1 machinery, 9.3ms
Speed: 3.4ms preprocess, 9.3ms inference, 4.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Hardhats, 4 Persons, 1 Safety Cone, 1 Safety V

In [28]:
# ==========================================================
# 🧠 7️⃣ Optional: Webcam Inference (For Local Only)
# ==========================================================
# Uncomment this section if running locally (not in Colab)
"""
def run_webcam(conf=0.5, imgsz=640):
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Webcam not detected")
        return
    print("✅ Press 'q' to quit")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        t0 = time.time()
        results = model(frame, conf=conf, imgsz=imgsz)
        t1 = time.time()
        fps = 1.0 / (t1 - t0)
        annotated = results[0].plot()
        cv2.putText(annotated, f"FPS: {fps:.2f}", (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow("Real-Time Detection", annotated)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

# run_webcam()
"""

'\ndef run_webcam(conf=0.5, imgsz=640):\n    cap = cv2.VideoCapture(0)\n    if not cap.isOpened():\n        print("❌ Webcam not detected")\n        return\n    print("✅ Press \'q\' to quit")\n    while True:\n        ret, frame = cap.read()\n        if not ret:\n            break\n        t0 = time.time()\n        results = model(frame, conf=conf, imgsz=imgsz)\n        t1 = time.time()\n        fps = 1.0 / (t1 - t0)\n        annotated = results[0].plot()\n        cv2.putText(annotated, f"FPS: {fps:.2f}", (20, 50),\n                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)\n        cv2.imshow("Real-Time Detection", annotated)\n        if cv2.waitKey(1) & 0xFF == ord(\'q\'):\n            break\n    cap.release()\n    cv2.destroyAllWindows()\n\n# run_webcam()\n'

In [31]:
# ==========================================================
# 📊 8️⃣ Generate Real-Time Inference Report
# ==========================================================
summary = {
    "Input": ["site_demo.mp4"],
    "Detection Quality": ["Good"],
    "Frame Rate (FPS)": [f"{avg_fps:.2f}"],
    "Avg Inference Time (ms)": [f"{avg_time*1000:.1f}"],
    "Observations": ["Stable detections with minor frame delay"]
}

report_df = pd.DataFrame(summary)
report_path_csv = f"{REPORT_DIR}/realtime_inference_report.csv"
report_df.to_csv(report_path_csv, index=False)

print("\n📈 Real-Time Inference Summary:")
print(report_df)
print("\n📄 Report saved to:", report_path_csv)


📈 Real-Time Inference Summary:
           Input Detection Quality Frame Rate (FPS) Avg Inference Time (ms)  \
0  site_demo.mp4              Good            38.21                    26.2   

                               Observations  
0  Stable detections with minor frame delay  

📄 Report saved to: /content/drive/MyDrive/SafetyEye/realtime_inference/report/realtime_inference_report.csv
